In [54]:

import numpy as np
import heapq
import pandas as pd
import os


In [55]:

# =====================================================
# SHARED CONSTANTS (same for both models)
# =====================================================

SIM_TIME             = 45 * 24
SPEED                = 20
AREA_SIZE            = 20
BASE_FARE            = 3
FARE_PER_MILE        = 2
PETROL_COST_PER_MILE = 0.20
CVAR_ALPHA           = 0.05
PATIENCE_RATE        = 5      # same in both — given in spec, not refitted

OUTPUT_DIR = "Comparison_1"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [56]:


# =====================================================
# MODEL A — GIVEN PARAMETERS (spec)
# =====================================================

GIVEN = {
    'driver_rate'   : 3,
    'rider_rate'    : 30,
    'patience_rate' : PATIENCE_RATE,

    # Uniform locations over [0, AREA_SIZE]
    'driver_loc' : lambda: np.array([
        np.random.uniform(0, AREA_SIZE),
        np.random.uniform(0, AREA_SIZE)
    ]),
    'pickup_loc' : lambda: np.array([
        np.random.uniform(0, AREA_SIZE),
        np.random.uniform(0, AREA_SIZE)
    ]),
    'dropoff_loc': lambda: np.array([
        np.random.uniform(0, AREA_SIZE),
        np.random.uniform(0, AREA_SIZE)
    ]),
}


In [57]:

# =====================================================
# MODEL B — MLE PARAMETERS (fitted from riders.xlsx)
# =====================================================

MLE = {
    'driver_rate'   : 4.74,
    'rider_rate'    : 34.6,
    'patience_rate' : PATIENCE_RATE,

    # Normal distributions fitted from real data
    'driver_loc' : lambda: np.array([
        np.clip(np.random.normal(9.9739,  4.3647), 0, AREA_SIZE),
        np.clip(np.random.normal(11.5133, 4.3362), 0, AREA_SIZE)
    ]),
    'pickup_loc' : lambda: np.array([
        np.clip(np.random.normal(8.3597,  4.2571), 0, AREA_SIZE),
        np.clip(np.random.normal(12.3175, 4.1904), 0, AREA_SIZE)
    ]),
    'dropoff_loc': lambda: np.array([
        np.clip(np.random.normal(11.2297, 4.5390), 0, AREA_SIZE),
        np.clip(np.random.normal(13.2626, 4.1690), 0, AREA_SIZE)
    ]),
}


In [58]:


# =====================================================
# DRIVER & RIDER
# =====================================================

class Driver:
    def __init__(self, did, loc, ou):
        self.id=did; self.location=loc; self.online_until=ou
        self.earnings=0; self.status='idle'; self.busy_time=0

class Rider:
    def __init__(self, rid, o, d, arr, ab):
        self.id=rid; self.origin=o; self.destination=d
        self.arrival_time=arr; self.abandon_time=ab; self.matched=False



In [59]:


# =====================================================
# SIMULATION — spec-correct, closest distance matching
# =====================================================

class BaselineSim:

    def __init__(self, params):
        self.params = params

        self.current_time     = 0
        self.event_list       = []
        self.idle_drivers     = {}
        self.busy_drivers     = {}
        self.waiting_riders   = {}
        self.driver_counter   = 0
        self.rider_counter    = 0
        self.total_riders     = 0
        self.abandoned_riders = 0
        self.total_queue_wait = 0
        self.total_true_wait  = 0
        self.driver_log       = []

    def exp_time(self, r): return np.random.exponential(1 / r)
    def dist(self, a, b):  return np.linalg.norm(np.array(a) - np.array(b))
    def ttime(self, d):
        mu = d / SPEED; return np.random.uniform(0.8 * mu, 1.2 * mu)
    def schedule(self, t, e, d=None):
        heapq.heappush(self.event_list, (t, e, d))

    # ── Spec-correct matching ────────────────────────────────────────
    def try_match(self, triggered_by, entity_id):
        if not self.idle_drivers or not self.waiting_riders:
            return

        if triggered_by == 'rider':
            # Rider arrived — find closest DRIVER to this rider
            rider    = self.waiting_riders[entity_id]
            best_did = min(self.idle_drivers,
                           key=lambda d: self.dist(self.idle_drivers[d].location,
                                                   rider.origin))
            best_rid = entity_id

        else:  # triggered_by == 'driver'
            # Driver freed up — find closest RIDER to this driver
            driver   = self.idle_drivers[entity_id]
            best_rid = min(self.waiting_riders,
                           key=lambda r: self.dist(driver.location,
                                                   self.waiting_riders[r].origin))
            best_did = entity_id

        driver = self.idle_drivers[best_did]
        rider  = self.waiting_riders[best_rid]
        rider.matched = True

        pd_ = self.dist(driver.location, rider.origin)
        td  = self.dist(rider.origin, rider.destination)
        pt  = self.ttime(pd_); tt2 = self.ttime(td); tt = pt + tt2

        qw = self.current_time - rider.arrival_time
        self.total_queue_wait += qw
        self.total_true_wait  += qw + pt

        earn = (BASE_FARE + FARE_PER_MILE * td) - \
               PETROL_COST_PER_MILE * (pd_ + td)

        driver.status = 'busy'; driver.busy_time += tt
        self.busy_drivers[best_did] = driver
        del self.idle_drivers[best_did]; del self.waiting_riders[best_rid]
        self.schedule(self.current_time + tt, 'DROPOFF',
                      (best_did, rider.destination, earn))

    def handle_driver_arrival(self):
        self.driver_counter += 1; did = self.driver_counter
        loc = self.params['driver_loc']()
        ou  = self.current_time + np.random.uniform(6, 8)
        self.idle_drivers[did] = Driver(did, loc, ou)
        self.driver_log.append({'driver_id': did, 'arrival_time': self.current_time,
                                'online_until': ou, 'earnings': 0, 'busy_time': 0})
        self.schedule(ou, 'DRIVER_OFFLINE', did)
        self.schedule(self.current_time + self.exp_time(self.params['driver_rate']),
                      'DRIVER_ARRIVAL')
        self.try_match(triggered_by='driver', entity_id=did)

    def handle_rider_arrival(self):
        self.rider_counter += 1; rid = self.rider_counter
        self.total_riders += 1
        o  = self.params['pickup_loc']()
        d  = self.params['dropoff_loc']()
        ab = self.current_time + self.exp_time(self.params['patience_rate'])
        self.waiting_riders[rid] = Rider(rid, o, d, self.current_time, ab)
        self.schedule(ab, 'RIDER_ABANDON', rid)
        self.schedule(self.current_time + self.exp_time(self.params['rider_rate']),
                      'RIDER_ARRIVAL')
        self.try_match(triggered_by='rider', entity_id=rid)

    def handle_dropoff(self, data):
        did, nloc, earn = data; driver = self.busy_drivers[did]
        driver.earnings += earn; driver.location = nloc; driver.status = 'idle'
        for d in self.driver_log:
            if d['driver_id'] == did:
                d['earnings'] += earn; d['busy_time'] = driver.busy_time; break
        del self.busy_drivers[did]
        if self.current_time < driver.online_until:
            self.idle_drivers[did] = driver
            self.try_match(triggered_by='driver', entity_id=did)

    def run(self):
        self.schedule(0, 'DRIVER_ARRIVAL'); self.schedule(0, 'RIDER_ARRIVAL')
        while self.event_list and self.current_time < SIM_TIME:
            self.current_time, et, data = heapq.heappop(self.event_list)
            if   et == 'DRIVER_ARRIVAL': self.handle_driver_arrival()
            elif et == 'RIDER_ARRIVAL':  self.handle_rider_arrival()
            elif et == 'DROPOFF':        self.handle_dropoff(data)
            elif et == 'RIDER_ABANDON':
                if data in self.waiting_riders:
                    self.abandoned_riders += 1; del self.waiting_riders[data]
            elif et == 'DRIVER_OFFLINE':
                self.idle_drivers.pop(data, None)
        return self.collect_results()

    def collect_results(self):
        matched = self.total_riders - self.abandoned_riders
        df      = pd.DataFrame(self.driver_log)
        df['online_duration']  = df['online_until'] - df['arrival_time']
        df['busy_time_capped'] = df[['busy_time','online_duration']].min(axis=1)
        df['idle_time']        = df['online_duration'] - df['busy_time_capped']
        df['idle_ratio']       = df['idle_time'] / df['online_duration']

        p  = df['earnings'].values; N = len(p); mp = p.mean()
        vp = np.sum((p - mp)**2) / (N - 1) if N > 1 else 0
        ps = np.sort(p)
        cvar     = np.mean(ps[:max(1, int(np.floor(CVAR_ALPHA * N)))])
        cvar_med = np.mean(ps[:max(1, int(np.floor(0.5 * N)))])

        return {
            'abandonment_rate'   : self.abandoned_riders / self.total_riders,
            'avg_queue_wait'     : self.total_queue_wait / matched if matched > 0 else 0,
            'avg_true_wait'      : self.total_true_wait  / matched if matched > 0 else 0,
            'matched_riders'     : matched,
            'total_riders'       : self.total_riders,
            'abandoned_riders'   : self.abandoned_riders,
            'avg_earnings'       : mp,
            'total_earnings'     : df['earnings'].sum(),
            'avg_profit_per_hr'  : df['earnings'].sum() / df['online_duration'].sum(),
            'avg_idle_time'      : df['idle_time'].mean(),
            'avg_idle_ratio'     : df['idle_ratio'].mean(),   # spec metric
            'variance'           : vp,
            'fairness_cv'        : np.sqrt(vp) / mp if mp > 0 else 0,
            'cvar'               : cvar,
            'delta_cvar_internal': cvar - cvar_med,
            '_driver_df'         : df,
        }



In [60]:

# =====================================================
# RUN BOTH MODELS
# =====================================================

configs = {
    'given' : GIVEN,
    'mle'   : MLE,
}

results = {}

for name, params in configs.items():
    print(f"Running {name.upper()} parameters...")
    np.random.seed(42)
    sim        = BaselineSim(params)
    res        = sim.run()
    results[name] = res
    res['_driver_df'].to_csv(f"{OUTPUT_DIR}/data_{name}.csv", index=False)
    print(f"  Riders      : {res['total_riders']}")
    print(f"  Matched     : {res['matched_riders']}")
    print(f"  Abandoned   : {res['abandoned_riders']}")
    print(f"  Abandon%    : {res['abandonment_rate']*100:.2f}%")
    print(f"  Queue Wait  : {res['avg_queue_wait']*60:.3f} min")
    print(f"  True Wait   : {res['avg_true_wait']*60:.3f} min")
    print(f"  Avg Earn£   : £{res['avg_earnings']:.2f}")
    print(f"  £/hr        : £{res['avg_profit_per_hr']:.2f}")
    print(f"  Idle time  : {res['avg_idle_time']:.2f}")
    print(f"  Idle Ratio  : {res['avg_idle_ratio']*100:.1f}%")
    print(f"  Variance    : {res['variance']:.2f}")
    print(f"  CVaR(5%)    : £{res['cvar']:.2f}")
    print()



Running GIVEN parameters...
  Riders      : 32289
  Matched     : 25055
  Abandoned   : 7234
  Abandon%    : 22.40%
  Queue Wait  : 2.010 min
  True Wait   : 24.303 min
  Avg Earn£   : £157.66
  £/hr        : £22.48
  Idle time  : 0.28
  Idle Ratio  : 3.9%
  Variance    : 637.61
  CVaR(5%)    : £104.89

Running MLE parameters...
  Riders      : 37420
  Matched     : 36864
  Abandoned   : 556
  Abandon%    : 1.49%
  Queue Wait  : 0.130 min
  True Wait   : 13.777 min
  Avg Earn£   : £116.20
  £/hr        : £16.64
  Idle time  : 2.54
  Idle Ratio  : 36.3%
  Variance    : 1344.69
  CVaR(5%)    : £30.45



In [62]:


# =====================================================
# SAVE SUMMARY CSV
# =====================================================

summary = pd.DataFrame([
    {
        'config'            : 'given',
        'driver_rate'       : GIVEN['driver_rate'],
        'rider_rate'        : GIVEN['rider_rate'],
        'location_model'    : 'uniform',
        'abandonment_pct'   : g['abandonment_rate'] * 100,
        'avg_queue_wait_min': g['avg_queue_wait'] * 60,
        'avg_true_wait_min' : g['avg_true_wait']  * 60,
        'matched_riders'    : g['matched_riders'],
        'total_riders'      : g['total_riders'],
        'avg_earnings'      : g['avg_earnings'],
        'avg_profit_per_hr' : g['avg_profit_per_hr'],
        'avg_idle_time'     : g['avg_idle_time'],
        'avg_idle_ratio'    : g['avg_idle_ratio'],
        'variance'          : g['variance'],
        'fairness_cv'       : g['fairness_cv'],
        'cvar'              : g['cvar'],
        'fairness_effect'   : 0.0,
        'delta_profit'      : 0.0,
        'delta_cvar'        : 0.0,
    },
    {
        'config'            : 'mle',
        'driver_rate'       : MLE['driver_rate'],
        'rider_rate'        : MLE['rider_rate'],
        'location_model'    : 'normal',
        'abandonment_pct'   : m['abandonment_rate'] * 100,
        'avg_queue_wait_min': m['avg_queue_wait'] * 60,
        'avg_true_wait_min' : m['avg_true_wait']  * 60,
        'matched_riders'    : m['matched_riders'],
        'total_riders'      : m['total_riders'],
        'avg_earnings'      : m['avg_earnings'],
        'avg_profit_per_hr' : m['avg_profit_per_hr'],
        'avg_idle_time'     : m['avg_idle_time'],
        'avg_idle_ratio'    : m['avg_idle_ratio'],
        'variance'          : m['variance'],
        'fairness_cv'       : m['fairness_cv'],
        'cvar'              : m['cvar'],
        'fairness_effect'   : fe,
        'delta_profit'      : delta_profit,
        'delta_cvar'        : delta_cvar,
    }
])

summary.to_csv(f"{OUTPUT_DIR}/summary_comparison.csv", index=False)
print(f"\nSaved to {OUTPUT_DIR}/")
print(f"  data_given.csv")
print(f"  data_mle.csv")
print(f"  summary_comparison.csv")


Saved to Comparison_1/
  data_given.csv
  data_mle.csv
  summary_comparison.csv
